# Workshop-Notebook: Grundlagen der KI-Agenten & Werkzeugnutzung

Dieses Notebook zeigt Ihnen Schritt für Schritt, wie Sie ein Sprachmodell (**Gemini**) mit einem **Werkzeug (Tool)** verbinden und eine einfache **ReAct-Schleife (Orchestrierung)** aufbauen.

**Event:** Digitaltag 2026 | *Mittelstand-Digital Zentrum Spreeland*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Unisvet/ki-agenten-intro/blob/main/notebooks/agent_demo.ipynb)

### Schritt 1: Installation & Setup
Zuerst installieren wir das offizielle Google GenAI SDK und konfigurieren unseren API-Schlüssel.

**API-Schlüssel:**
Holen Sie sich Ihren kostenlosen Gemini API-Schlüssel von [Google AI Studio](https://aistudio.google.com/app/api-keys).
Führen Sie die Code-Zellen unten nacheinander aus. Bei der zweiten Zelle fügen Sie Ihren kopierten Schlüssel einfach ein (die Eingabe wird aus Sicherheitsgründen maskiert) und drücken **Enter**.

In [ ]:
# Installation der SDKs und Upgrade von Protobuf zur Vermeidung von Inkompatibilitäten
%pip install --upgrade protobuf google-genai google-adk

In [ ]:
import os
import getpass
from google import genai
from google.genai import types
from google.genai.errors import APIError

# 1. Frage den Nutzer direkt nach dem Gemini API-Key (Eingabe wird maskiert)
api_key = getpass.getpass("Bitte fügen Sie Ihren Gemini API-Key ein und drücken Sie Enter: ")
os.environ["GEMINI_API_KEY"] = api_key.strip()

# 2. Initialisierung des GenAI Clients
client = genai.Client()
model_id = 'gemini-3.5-flash'
print("Gemini API-Schlüssel wurde erfolgreich konfiguriert und der Client initialisiert!")

### Schritt 2: Einfache Modellanfrage
Bevor wir mit Werkzeugen (Tools) arbeiten, testen wir, ob wir das Sprachmodell direkt anfragen können. Wir stellen eine einfache Frage wie: *"Was ist ein KI-Agent?"* und formatieren die Antwort für bessere Lesbarkeit.

In [ ]:
import textwrap

# Einfache Anfrage an Gemini senden
response = client.models.generate_content(
    model=model_id,
    contents="Was ist ein KI-Agent? Beantworten Sie in 2 bis 3 kurzen Sätzen."
)

print("Antwort von Gemini:\n")
# Formatierter Zeilenumbruch für bessere Lesbarkeit im Notebook
print(textwrap.fill(response.text, width=80))

### Schritt 3: Echte API als Werkzeug definieren (Tool)
Nun wollen wir dem Modell Zugriff auf Echtzeit-Informationen geben. Wir definieren eine Python-Funktion, die die **freie Open-Meteo-Wetter-API** abfragt, um aktuelle Wetterdaten für Cottbus zu ermitteln (ohne API-Key).

In [ ]:
import urllib.request
import json

def get_wetter_cottbus(datum: str) -> str:
    """Fragt das aktuelle Wetter für Cottbus an einem bestimmten Datum ab.
    
    Args:
        datum: Das gewünschte Datum im Format JJJJ-MM-TT (z. B. 2026-06-26 oder heute/morgen)
    """
    # Cottbus Koordinaten
    lat = 51.7563
    lon = 14.3329
    url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&daily=temperature_2m_max,temperature_2m_min,precipitation_probability_max&timezone=Europe/Berlin"
    
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as response:
            data = json.loads(response.read().decode())
        
        daily = data.get("daily", {})
        time_list = daily.get("time", [])
        
        if datum in time_list:
            idx = time_list.index(datum)
            temp_max = daily.get("temperature_2m_max", [])[idx]
            temp_min = daily.get("temperature_2m_min", [])[idx]
            rain_prob = daily.get("precipitation_probability_max", [])[idx]
            return (f"Wettervorhersage für Cottbus am {datum}: "
                    f"Höchsttemperatur {temp_max}°C, Tiefsttemperatur {temp_min}°C, "
                    f"Regenwahrscheinlichkeit {rain_prob}%.")
        else:
            return (f"Für das Datum {datum} liegen leider keine aktuellen 7-Tage-Vorhersagedaten vor. "
                    f"Bitte wählen Sie ein Datum im Bereich {time_list[0]} bis {time_list[-1]}.")
    except Exception as e:
        return f"Fehler bei der Wetterabfrage: {str(e)}. (Ersatzwert: 20 Grad, leicht bewölkt)."

### Schritt 4: Modell mit Werkzeug verbinden (Tool Binding)
Wir senden eine Anfrage, die Wetterdaten benötigt. Wir generieren das Datum für **morgen** dynamisch, damit die Abfrage immer in der 7-Tage-Vorhersage der API liegt.

In [ ]:
from datetime import datetime, timedelta

# Dynamisches Datum für morgen generieren (Format: JJJJ-MM-TT)
morgen = (datetime.now() + timedelta(days=1)).strftime("%Y-%m-%d")
prompt = f"Brauche ich am {morgen} in Cottbus eine Regenjacke?"
print(f"Anfrage an das Modell: {prompt}\n")

response = client.models.generate_content(
    model=model_id,
    contents=prompt,
    config=types.GenerateContentConfig(
        # Hier registrieren wir unsere Funktion als Werkzeug
        tools=[get_wetter_cottbus],
        temperature=0,
        # Wir deaktivieren das automatische Ausführen, um den rohen Funktionsaufruf zu sehen
        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True)
    )
)

# Das Modell führt die Funktion nicht selbst aus, sondern fordert uns dazu auf:
print("Funktionsaufrufe des Modells:", response.function_calls)

### Schritt 5: Die Orchestrierungsschleife (ReAct Loop)
Mit `automatic_function_calling` überlassen wir dem SDK die Ausführung der Funktion im Hintergrund. Der Agent holt sich die Wetterdaten von der API und formuliert die finale Antwort, die wir wieder passend umbrechen.

In [ ]:
import textwrap
from datetime import datetime, timedelta

morgen = (datetime.now() + timedelta(days=1)).strftime("%Y-%m-%d")
prompt = f"Brauche ich am {morgen} in Cottbus eine Regenjacke? Beantworte basierend auf dem Wetterbericht."

response_auto = client.models.generate_content(
    model=model_id,
    contents=prompt,
    config=types.GenerateContentConfig(
        tools=[get_wetter_cottbus],
        automatic_function_calling=types.AutomaticFunctionCallingConfig(maximum_remote_calls=3)
    )
)

print("Endgültige Antwort des Agenten:\n")
print(textwrap.fill(response_auto.text, width=80))

### Schritt 6: Google Agent Development Kit (ADK) (Optional)
Das **Google Agent Development Kit (ADK)** (`google-adk`) ist ein code-first Framework zum Erstellen, Debuggen und Ausführen von KI-Agenten. Da es sich um eine reine Python-Bibliothek handelt, läuft es im Gegensatz zu komplexeren, betriebssystemnahen SDKs vollkommen problemlos in Google Colab.

Hier konfigurieren wir einen einfachen ADK-Agenten und führen ihn mit einem `InMemoryRunner` aus.

In [ ]:
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.genai import types
import textwrap

# 1. Agenten definieren
my_agent = Agent(
    name="mdz_assistant",
    model="gemini-3.5-flash",
    instruction="Du bist ein hilfreicher KI-Assistent. Beantworte alle Fragen freundlich und präzise."
)

# 2. Runner initialisieren und Session-Erstellung aktivieren
runner = InMemoryRunner(agent=my_agent)
runner.auto_create_session = True

# 3. Anfrage vorbereiten
prompt = "Schreibe ein kurzes Gedicht über Cottbus und die Lausitz."
msg = types.Content(parts=[types.Part(text=prompt)])

# 4. Agenten ausführen
events = runner.run(
    user_id="workshop_user",
    session_id="session_123",
    new_message=msg
)

print("Antwort des ADK-Agenten:\n")
for event in events:
    # Der Runner liefert Events. Wir geben den Text aus dem Content-Event aus.
    if hasattr(event, "content") and event.content:
        for part in event.content.parts:
            if part.text:
                print(textwrap.fill(part.text, width=80))

### Schritt 7: ADK-Agent mit integriertem Web-Suchwerkzeug (Google Search)
Das ADK bietet eingebaute Werkzeuge wie `google_search`. Hier konfigurieren wir einen Forschungsagenten (`research_agent`), der die Google-Suche nutzen kann, um aktuelle Informationen abzufragen, bevor er antwortet.

In [ ]:
import asyncio
from google.adk.agents import Agent
from google.adk.tools import google_search
from google.adk.runners import Runner
from google.adk.sessions.in_memory_session_service import InMemorySessionService
from google.genai import types
import textwrap

# 1. Forschungs-Agent definieren
research_agent = Agent(
    name="research_agent",
    model=model_id, # verwendet das zuvor definierte 'gemini-3.5-flash'
    description="Ein Agent, der im Internet nach aktuellen Themen recherchieren kann.",
    instruction="Du bist ein hilfreicher Forschungsassistent. Nutze das google_search Werkzeug, um aktuelle Informationen zu finden, bevor du antwortest.",
    tools=[google_search]
)

# 2. Standard-Laufzeitumgebung einrichten
session_service = InMemorySessionService()
session = await session_service.get_session(app_name="colab-app", user_id="workshop_user", session_id="colab-test-session")
if not session:
    session = await session_service.create_session(app_name="colab-app", user_id="workshop_user", session_id="colab-test-session")
runner = Runner(app_name="colab-app", agent=research_agent, session_service=session_service)

# 3. Asynchrone Hilfsfunktion zur Ausführung
async def ask_agent(query_text):
    print(f"User: {query_text}\n")
    
    # Anfrage an den Runner senden
    msg = types.Content(parts=[types.Part(text=query_text)])
    
    async for event in runner.run_async(new_message=msg, user_id="workshop_user", session_id=session.id):
        # Identifiziere und drucke das finale Antwort-Event des Agenten
        if event.is_final_response() and event.content:
            text = "".join(part.text for part in event.content.parts if part.text)
            print("Agent Response:\n")
            print(textwrap.fill(text, width=80))

# Hilfsfunktion direkt in der Zelle ausführen
await ask_agent("Was sind die neuesten Entdeckungen des James-Webb-Weltraumteleskops?")